<a href="https://colab.research.google.com/github/suhailabdulwasi/ds2002-fa26/blob/main/notebooks/03-pandas-cleaning/2026-09-18%20%E2%80%94%20Pandas%20Challenge%20%E2%80%94%20Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Pandas Challenge

**Lab — 2026-09-18 · Fall 2026**  

---

## Lab 04 — Pandas Challenge

Four hundred generated orders. Each question builds toward a demand report you could hand a vendor.

The data is seeded, so everyone's numbers should match. That is deliberate: if your total revenue differs from your neighbor's, one of you has a bug, and the assertions at the end will tell you which.

Every answer needs the number **and** a sentence saying what it means. A cell that prints `4218.5` with no interpretation is half an answer.

In [9]:
import pandas as pd, numpy as np
rng = np.random.default_rng(4)
n = 400
df = pd.DataFrame({
    'vendor_id': rng.choice(['V-01','V-05','V-10','V-18'], n),
    'category': rng.choice(['Food','Merch','RainGear','Drink'], n, p=[.5,.2,.1,.2]),
    'qty': rng.integers(1, 4, n),
    'price': rng.choice([4.5, 6.0, 7.5, 12.0, 24.0], n),
})
df.head()

,vendor_id,category,qty,price
0,V-10,Drink,2,24.0
1,V-18,RainGear,1,12.0
2,V-18,Drink,3,4.5
3,V-10,Food,2,12.0
4,V-18,Drink,3,7.5


### Q1 — Add `revenue`, then report total revenue and total units.

*Expected: 400 rows, and revenue should land between $8,000 and $9,000.*

In [11]:
df['revenue'] = df['qty'] * df['price']

total_revenue = df['revenue'].sum()
total_units = df['qty'].sum()

print("Total revenue:", total_revenue)
print("Total units:", total_units)

Total revenue: 8520.0
Total units: 783


### Q2 — Revenue by category, highest to lowest.

Include the share of total as a percentage in the same table.

In [12]:
by_category = (
    df.groupby('category', as_index=False)['revenue']
    .sum()
)

by_category['share_pct'] = (
    by_category['revenue'] / df['revenue'].sum() * 100
).round(1)

by_category = by_category.sort_values('revenue', ascending=False)

by_category

,category,revenue,share_pct
1,Food,4293.0,50.4
2,Merch,1771.5,20.8
0,Drink,1554.0,18.2
3,RainGear,901.5,10.6


### Q3 — Which vendor has the highest *average* order revenue?

Report the average alongside the order count for each vendor. A high average on twelve orders is a different claim from a high average on two hundred.

In [13]:
by_vendor = (
    df.groupby('vendor_id')
    .agg(
        avg_order_revenue=('revenue', 'mean'),
        order_count=('revenue', 'size')
    )
    .sort_values('avg_order_revenue', ascending=False)
)

by_vendor

,avg_order_revenue,order_count
vendor_id,,
V-01,22.595745,94
V-18,21.750000,108
V-05,20.580645,93
V-10,20.314286,105


### Q4 — What share of revenue comes from Merch?

Print it as a percentage rounded to one decimal.

In [14]:
merch_share = (
    df.loc[df['category'] == 'Merch', 'revenue'].sum()
    / df['revenue'].sum()
    * 100
)

print(f"Merch revenue share: {merch_share:.1f}%")

Merch revenue share: 20.8%


### Q5 — Join in the vendor names.

The frame only has `vendor_id`. Merge the lookup below so your report is readable.

**Requirements:** left join, `validate='many_to_one'`, and prove the row count and revenue total did not change. One vendor id in the orders is not in this lookup — find it, and decide what to do about it.

In [23]:
vendors = pd.DataFrame({
    'vendor_id': ['V-01', 'V-05', 'V-10'],
    'vendor_name': ['Hops Burgers', 'Rotunda Tacos', 'Cav Merch North'],
})

joined = df.merge(
    vendors,
    on='vendor_id',
    how='left',
    validate='many_to_one'
)

print("Rows before merge:", len(df))
print("Rows after merge:", len(joined))
print("Revenue before merge:", df['revenue'].sum())
print("Revenue after merge:", joined['revenue'].sum())

Rows before merge: 400
Rows after merge: 400
Revenue before merge: 8520.0
Revenue after merge: 8520.0


**The unmatched vendor, and what I did about it: V-18 was the unmatched vendor. I kept its orders in the report and labeled the vendor as “Unknown (V-18)” so its revenue would not be excluded from the analysis.

### Q6 — A pivot table: vendors down the side, categories across the top, revenue in the cells.

Add row and column totals so it reads as a report rather than a grid of numbers.

In [27]:
# Keep the unmatched vendor in the report
joined['vendor_name'] = joined['vendor_name'].fillna('Unknown (V-18)')

# Create pivot table
pivot = pd.pivot_table(
    joined,
    index='vendor_name',
    columns='category',
    values='revenue',
    aggfunc='sum',
    fill_value=0,
    margins=True,
    margins_name='Total'
)

pivot

category,Drink,Food,Merch,RainGear,Total
vendor_name,,,,,
Cav Merch North,502.5,1054.5,400.5,175.5,2133.0
Hops Burgers,171.0,1338.0,373.5,241.5,2124.0
Rotunda Tacos,298.5,882.0,489.0,244.5,1914.0
Unknown (V-18),582.0,1018.5,508.5,240.0,2349.0
Total,1554.0,4293.0,1771.5,901.5,8520.0


### Q7 — Validate your work

**TODO:** uncomment and make these pass. Assign your results to the named variables as you go.

In [29]:
assert len(df) == 400
assert 8000 < df['revenue'].sum() < 9000
assert abs(by_category['revenue'].sum() - df['revenue'].sum()) < 0.01
assert len(joined) == len(df), "the vendor merge changed the row count"

print("checks passed.")

checks passed.


### Write-up

**a)** What would you tell these vendors to do differently next game? One paragraph, with at least two numbers from your report in it.

**b)** Which of your seven answers is the least trustworthy, and why? Point at a specific weakness — a small group size, an unmatched vendor, a category that is really two things.

A. I will advise the vendors to concentrate more on those items and categories of items that were responsible for the majority of the revenues. On the whole, the vendors realized revenues worth $8,520 from 783 units sold. They can use the category table to determine the top-selling items and then bring more such items to the next game. In addition, they may consider the poorly performing categories and change pricing and marketing for such items. Such approach will help them sell more items and increase their revenues.

B.The vendor analysis is the least reliable one since there is no corresponding vendor name for V-18 in the lookup table. Though the left join retained all 400 orders along with the total revenue of $8,520, we still cannot assign the sales of V-18 to any particular vendor until the lookup data is fixed.